In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore") # Suppress warnings for cleaner output

In [36]:
def load_and_standardize(path, date_cols=None):
    """
    Load a CSV file into a pandas DataFrame and standardize column names.
    
    Parameters:
    - path (str): Path to the CSV file.
    - date_cols (list of str, optional): List of columns to parse as dates.
    
    Returns:
    - pd.DataFrame: Loaded and standardized DataFrame.
    """
    df = pd.read_csv(path, parse_dates=date_cols)
    df.columns = (
        df.columns.str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    
     # Convert datetime columns to UTC
    if date_cols: # only if date_cols=[timestamp] is provided
        for col in date_cols:
            df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
    
      # Convert numeric columns robustly
    for col in df.select_dtypes(include=["object"]).columns:
        # pandas convertit les nombres en object s'ils
        # contiennent des valeurs manquantes ou des chaînes non numériques 
        # donc include = ["object"]
        if any(df[col].astype(str).str.match(r"^\d+(\.\d+)?$")):
            # any si au moins une valeur correspond au format numérique
            # astype(str) je convertis d'abord tout en string pour éviter les erreurs
            # sur les NaN, None ou N/A ;
            # puis match avec r"^\d+(\.\d+)?$" qui signifie :
            # début de chaîne (^), un ou plusieurs chiffres (\d+),
            # optionnellement suivis d'un point et d'un ou plusieurs chiffres ((\.\d+)?),
            # fin de chaîne ($)
            df[col] = pd.to_numeric(df[col], errors="ignore")
            # errors="ignore" pour éviter les erreurs et laisser les 
            # autres valeurs telles quelles
            # si la conversion échoue
            # to_numeric convertit en int ou float selon les données
            
    return df  # attention à la bonne indentation!

In [37]:
user_profiles = load_and_standardize("user_profiles.csv")
content = load_and_standardize("content.csv")
viewing_events = load_and_standardize("viewing_events.csv", date_cols=["timestamp"])
streaming_events = load_and_standardize("streaming_events.csv", date_cols=["timestamp"])


In [38]:
missing_users = set(viewing_events.user_id) - set(user_profiles.user_id)
missing_content = set(viewing_events.content_id) - set(content.content_id)

print("Missing users from viewing_events in user_profiles:", len(missing_users))
print("Missing content from missing_contents in user_profiles:", len(missing_content))


Missing users from viewing_events in user_profiles: 0
Missing content from missing_contents in user_profiles: 0


# tache 2

In [39]:
viewing_events["session_date"] = viewing_events["timestamp"].dt.date
# je recrée une colonne session_date pour extraire le jour dans le timestamp
# dt pour vectoriser et date pour extraire


In [40]:
sessions_df = (viewing_events
    .groupby(["user_id", "content_id", "session_date"])
    .agg(
        watched_seconds=("watched_seconds", "sum"),
        total_seconds=("total_seconds", "max")
    ).reset_index()
)
# reset_index() pour remettre user_id, content_id et 
# session_date en colonnes et non en multi_index

In [41]:
sessions_df["completion_rate"] = (
    sessions_df["watched_seconds"] / sessions_df["total_seconds"]
).clip(0,1)
    

In [42]:
sessions_df = (
    sessions_df.merge(
        content, on="content_id", how="left"
    )
)
sessions_df = sessions_df.merge(user_profiles, on="user_id", how="left")


In [43]:
labels = ["Very Short", "Short", "Medium", "Long", "Very Long"]

sessions_df["duration_bucket"] = pd.qcut(
    sessions_df["total_seconds"],
    q=5, # découper l'échantillon en 5 quantiles égaux avec qcut # cut pour des intervalles fixes # à revoir
    labels=labels,
    duplicates="drop" # si pas assez de donnees pour 5 quantiles distincts
    # on reduit le nombre de quantiles pour éviter des erreurs de plantage
)
print(sessions_df.head())


    user_id   content_id session_date  watched_seconds  total_seconds  \
0  user_001  content_006   2024-02-01             2552           3895   
1  user_001  content_016   2024-02-19              225           2385   
2  user_001  content_020   2024-02-08              619           1871   
3  user_001  content_022   2024-02-28              902           3872   
4  user_001  content_028   2024-02-28              785           4056   

   completion_rate        genre primary_genre content_type  rating  \
0         0.655199  documentary   documentary       series     7.6   
1         0.094340     thriller      thriller  documentary     7.6   
2         0.330839       sci-fi        sci-fi        movie     6.4   
3         0.232955       comedy        comedy       series     7.1   
4         0.193540      romance       romance        movie     9.0   

  subscription_type  user_age country  monthly_revenue duration_bucket  
0           premium        32      DE            13.99            L

# tache 3

In [44]:
genre_stats = (
    sessions_df
    .groupby("primary_genre")
    .agg(
        completion_rate=("completion_rate", "mean"),
        session_count=("completion_rate", "count")
        # count en plus pour éviter des moyennes trompeuses de catégories ayant peu de sessions
    )
    .sort_values("completion_rate", ascending=False) 
).reset_index()
print(genre_stats)


  primary_genre  completion_rate  session_count
0         drama         0.258138            103
1       romance         0.240064            128
2        sci-fi         0.239996            162
3        horror         0.221052             94
4        action         0.219030             64
5      thriller         0.205333            218
6        comedy         0.202865            133
7   documentary         0.200805             98


In [45]:
rank_df = (
    sessions_df
    .groupby(["country", "content_id"])
    .agg(watched_seconds=("watched_seconds", "sum"))
    .reset_index()
    # première étape : somme du temps de visionnage sur 2 clés de regroupement : watched_seconds par pays et par contenu
)

rank_df["rank"] = (
    rank_df.groupby("country")["watched_seconds"]
    # rappel syntaxe : df.groupby("col1")["col2"] pour grouper par col1 et appliquer une fonction sur col2
    # différente de agg qui agrège en une seule ligne par groupe
    .rank(method="first", ascending=False)
    # 2e étape : classement par pays en fonction du temps de visionnage total
    # method "first" pour attribuer des rangs en fonction de l'ordre d'apparition en cas d'égalité
)

top10 = rank_df[rank_df["rank"] <= 10].sort_values(["country", "rank"])
# filtrer pour ne garder que les top 10 par pays 
# le tri multi-colonnes suit l'ordre des colonnes dans la liste
# tri par pays puis à l'intérieur de chaque pays par rang
print(top10)

    country   content_id  watched_seconds  rank
29       BR  content_035             7333   1.0
23       BR  content_029             6965   2.0
42       BR  content_049             6653   3.0
82       BR  content_092             6512   4.0
81       BR  content_091             6265   5.0
..      ...          ...              ...   ...
450      US  content_006             4522   6.0
461      US  content_020             4139   7.0
489      US  content_061             4075   8.0
498      US  content_074             3796   9.0
464      US  content_023             3337  10.0

[70 rows x 4 columns]


In [46]:
subset = sessions_df[sessions_df.subscription_type.isin(["basic", "premium"])]
# filtrer pour ne garder que les abonnements basic et premium avec isin
# sessions_df.subscription_type.isin(["basic", "premium"]) renvoie un masque booléen true/false
# ensuite sessions_df[masque] pour filtrer les lignes où le masque est == True
# pas besoin de .loc car filtre uniquement sur les lignes
# équivalent à : subset = sessions_df[(sessions_df.subscription_type == "basic") | (sessions_df.subscription_type == "premium")]
# ou subset = sessions_df.query("subscription_type in ['basic', 'premium']")
# ou subset=[value in ["basic","premium"] for value in sessions_df.subscription_type]



perf = (
    subset
    .groupby(["country", "subscription_type"])
    .completion_rate.mean()
    # pareil que groupby(["country", "subscription_type"])["completion_rate"].mean()
    # ou groupby(["country", "subscription_type"]).agg(completion_rate=("completion_rate", "mean"))
    .unstack()
    # rappel : unstack pour transformer les valeurs uniques d'une colonne en colonnes
    # ici pour avoir une colonne par type d'abonnement basic et premium
    # au lieu d'avoir une ligne par pays et par type d'abonnement
)

underperform = perf[perf["premium"] < perf["basic"]]
print(underperform)
# tableau des pays où le taux de complétion des abonnés premium est inférieur à celui des abonnés basic


subscription_type     basic   premium
country                              
BR                 0.234241  0.228048
JP                 0.230103  0.224401
US                 0.239188  0.216841


# tache 4

In [47]:
sessions_df.to_csv("sessions.csv", index=False)
print("✓ Exported sessions.csv")


✓ Exported sessions.csv


In [48]:
print("Shape:", sessions_df.shape)
print(sessions_df.completion_rate.describe())
sessions_df.sample(10)


Shape: (1000, 15)
count    1000.000000
mean        0.222415
std         0.182617
min         0.006198
25%         0.082054
50%         0.172825
75%         0.314976
max         0.885021
Name: completion_rate, dtype: float64


,user_id,content_id,session_date,watched_seconds,total_seconds,completion_rate,genre,primary_genre,content_type,rating,subscription_type,user_age,country,monthly_revenue,duration_bucket
343,user_020,content_002,2024-01-25,1138,2582,0.440744,documentary,documentary,documentary,9.5,basic,66,BR,17.99,Medium
945,user_048,content_088,2024-01-04,111,1097,0.101185,drama,drama,series,7.9,standard,58,FR,13.99,Very Short
682,user_036,content_042,2024-02-22,837,4243,0.197266,thriller,thriller,documentary,9.1,premium,57,BR,17.99,Long
259,user_015,content_043,2024-03-17,68,2568,0.026480,comedy,comedy,documentary,9.3,standard,44,BR,8.99,Short
722,user_037,content_080,2024-01-17,639,4062,0.157312,horror,horror,series,6.7,premium,46,JP,13.99,Long
705,user_037,content_026,2024-03-10,1289,1742,0.739954,horror,horror,series,6.8,premium,46,JP,13.99,Very Short
24,user_002,content_049,2024-03-25,145,829,0.174910,drama,drama,series,6.1,standard,68,CA,8.99,Very Short
745,user_039,content_024,2024-03-19,791,3176,0.249055,action,action,movie,6.5,standard,23,US,17.99,Medium
604,user_032,content_062,2024-01-01,146,1022,0.142857,romance,romance,series,7.9,standard,64,JP,13.99,Very Short
169,user_009,content_076,2024-01-12,311,4234,0.073453,documentary,documentary,documentary,8.9,basic,58,US,8.99,Long
